In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Singularity Machine Learning - Classification: פונקציית Qiskit מאת Multiverse Computing
> **Note:** * פונקציות Qiskit הן פיצ'ר ניסיוני הזמין רק למשתמשי IBM Quantum&reg; Premium Plan, Flex Plan, ו-On-Prem (דרך IBM Quantum Platform API) Plan. הן בסטטוס גרסת תצוגה מקדימה וכפופות לשינויים.

## סקירה כללית
עם פונקציית "Singularity Machine Learning - Classification", אתה יכול לפתור בעיות למידת מכונה מהעולם האמיתי על חומרה קוונטית מבלי שתידרש מומחיות קוונטית. פונקציית Application זו, המבוססת על שיטות אנסמבל, היא מסווג היברידי. היא מנצלת שיטות קלאסיות כמו boosting, bagging ו-stacking לאימון אנסמבל ראשוני. לאחר מכן, אלגוריתמים קוונטיים כגון variational quantum eigensolver (VQE) ו-quantum approximate optimization algorithm (QAOA) מוחלים כדי לשפר את הגיוון, יכולות ההכללה, והמורכבות הכוללת של האנסמבל המאומן.

שלא כמו פתרונות קוונטיים אחרים ללמידת מכונה, פונקציה זו מסוגלת לטפל בסטי נתונים בקנה מידה גדול עם מיליוני דוגמאות ותכונות, מבלי שתהיה מוגבלת על ידי מספר ה-Qubits ב-QPU המטרה. מספר ה-Qubits קובע רק את גודל האנסמבל שניתן לאמן. בנוסף, הפונקציה גמישה מאוד, וניתן להשתמש בה לפתרון בעיות סיווג במגוון רחב של תחומים, כולל פיננסים, בריאות וסייבר.
היא משיגה באופן עקבי דיוקים גבוהים בבעיות מאתגרות קלאסית הכוללות סטי נתונים בממד גבוה, רועשים וחסרי איזון.
![כיצד זה עובד](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
היא נבנתה עבור:
1. מהנדסים ומדעני נתונים בחברות המבקשים לשפר את ההצעות הטכנולוגיות שלהם על ידי שילוב למידת מכונה קוונטית במוצרים ובשירותים שלהם,
2. חוקרים במעבדות מחקר קוונטי החוקרים יישומי למידת מכונה קוונטית ומבקשים לנצל את המחשוב הקוונטי למשימות סיווג, ו-
3. סטודנטים ומורים במוסדות חינוך בקורסים כמו למידת מכונה, המבקשים להדגים את יתרונות המחשוב הקוונטי.

הדוגמה הבאה מציגה את הפונקציונליות השונות שלה, כולל `create`, `list`, `fit` ו-`predict`, ומדגימה את השימוש בה בבעיה סינתטית המורכבת משני חצאי עיגולים משתלבים — בעיה ידועה לשמצה בגלל הגבול ההחלטה הלא-לינארי שלה.


## תיאור הפונקציה
פונקציית Qiskit זו מאפשרת למשתמשים לפתור בעיות סיווג בינארי באמצעות מסווג האנסמבל המשופר קוונטית של Singularity. מאחורי הקלעים, היא משתמשת בגישה היברידית לאימון קלאסי של אנסמבל מסווגים על סט הנתונים המתויג, ולאחר מכן לאופטימיזציה שלו לגיוון מרבי והכללה באמצעות Quantum Approximate Optimization Algorithm (QAOA) על IBM&reg; QPUs. דרך ממשק ידידותי למשתמש, ניתן להגדיר מסווג בהתאם לדרישות שלך, לאמן אותו על סט הנתונים שתבחר, ולהשתמש בו לביצוע תחזיות על סט נתונים שלא נראה בעבר.

לפתרון בעיית סיווג גנרית:
1. עבד מראש את סט הנתונים, וחלק אותו לסטי אימון ובדיקה. באופן אופציונלי, ניתן לחלק עוד יותר את סט האימון לסטי אימון ואימות. ניתן להשיג זאת באמצעות [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html).
2. אם סט האימון אינו מאוזן, ניתן לדגום אותו מחדש כדי לאזן את המחלקות באמצעות [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets).
3. העלה את סטי האימון, האימות והבדיקה בנפרד לאחסון הפונקציה באמצעות שיטת `file_upload` של הקטלוג, תוך העברת הנתיב הרלוונטי בכל פעם.
4. אתחל את המסווג הקוונטי באמצעות פעולת `create` של הפונקציה, המקבלת hyperparameters כגון מספר סוגי הלומדים, הרגולריזציה (ערך lambda), ואפשרויות אופטימיזציה כולל מספר שכבות, סוג האופטימייזר הקלאסי, ה-Backend הקוונטי, וכן הלאה.
5. אמן את המסווג הקוונטי על סט האימון באמצעות פעולת `fit` של הפונקציה, תוך העברת סט האימון המתויג, וסט האימות אם רלוונטי.
6. בצע תחזיות על סט הבדיקה שלא נראה בעבר באמצעות פעולת `predict` של הפונקציה.
## גישה מבוססת פעולות
הפונקציה משתמשת בגישה מבוססת פעולות. ניתן לחשוב עליה כסביבה וירטואלית שבה אתה משתמש בפעולות לביצוע משימות או לשינוי מצבה. כרגע, היא מציעה את הפעולות הבאות: [list](#1-list), [create](#2-create), [delete](#3-delete), [fit](#4-fit), [predict](#5-predict), [fit_predict](#6-fit-predict) ו-[create_fit_predict](#7-create-fit-predict). הדוגמה הבאה מדגימה את פעולת `create_fit_predict`.

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

## Examples

### Classify a dataset

In this example, you'll use the "Singularity Machine Learning - Classification" function to classify a dataset consisting of two interleaving, moon-shaped half-circles. The dataset is synthetic, two-dimensional, and labeled with binary labels. It is created to be challenging for algorithms such as centroid-based clustering and linear classification.

![Moons dataset](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)

Through this process, you'll learn how to create the classifier, fit it to the training data, use it to predict on the test data, and delete the classifier when you're finished.

Before starting, you need to install [scikit-learn](https://scikit-learn.org/). Install it using the following command:

```sh
python3 -m pip install scikit-learn
```

Perform the following steps:

1) Create the synthetic dataset using the [`make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) function from [scikit-learn](https://scikit-learn.org/).
2) Upload the generated synthetic dataset to the [shared data directory](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).
3) Create the quantum-enhanced classifier using the [`create`](/docs/api/functions/multiverse-computing-singularity#create) action.
4) Enlist your classifiers using the [`list`](/docs/api/functions/multiverse-computing-singularity#list) action.
5) Train the classifier on the train data using the [`fit`](/docs/api/functions/multiverse-computing-singularity#fit) action.
6) Use the trained classifier to predict on the test data using the [`predict`](/docs/api/functions/multiverse-computing-singularity#predict) action.
7) Delete the classifier using the [`delete`](/docs/api/functions/multiverse-computing-singularity#delete) action.
8) Clean up after you're done.

**Step 1.** Import the necessary modules and generate the synthetic dataset, then split it into training and test datasets.

In [2]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[ 0.84757037 -0.48831433]
 [ 0.98132552  0.19235443]
 [-0.71626723  0.6978261 ]
 [ 1.18957848 -0.48186557]
 [ 0.52118982 -0.37791846]
 [ 0.81115408  0.58483251]
 [ 0.48706462  0.87336593]
 [-0.81880144  0.57407682]
 [ 1.67335408 -0.23932015]
 [ 0.50181306  0.8649761 ]]
Targets: [1 0 0 1 1 0 0 0 1 0]


### 1. List

פעולת `list` מאחזרת את כל המסווגים המאוחסנים בפורמט `*.pkl.tar` מספריית הנתונים המשותפת. ניתן גם לגשת לתוכן הספרייה הזו באמצעות שיטת `catalog.files()`. בדרך כלל, פעולת ה-list מחפשת קבצים עם הסיומת `*.pkl.tar` בספריית הנתונים המשותפת ומחזירה אותם בפורמט רשימה.
#### קלט
|     שם    |    סוג     | תיאור |   חובה  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | שם הפעולה מבין `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` ו-`delete`. | כן |

#### שימוש

In [3]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


### 2. Create

פעולת `create` יוצרת מסווג מסוג `quantum_classifier` שצוין באמצעות הפרמטרים שסופקו, ושומרת אותו בספריית הנתונים המשותפת.

> **Note:** הפונקציה תומכת כרגע רק ב-`QuantumEnhancedEnsembleClassifier`.
#### קלט
|     שם    |    סוג     | תיאור | חובה  | ברירת מחדל |
|-------------|-------------|-------------|-----------|---------|
| `action` | `str` | שם הפעולה מבין `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` ו-`delete`. | כן | - |
| `name` | `str` | שם המסווג הקוונטי, למשל `spam_classifier`. | כן | - |
| `instance` | `str` | מופע IBM. | כן | - |
| `backend_name` | `str` | משאב מחשוב IBM. ברירת המחדל היא `None`, כלומר ה-Backend עם הכי מעט משימות ממתינות ייבחר. | לא | `None` |
| `quantum_classifier` | `str` | סוג המסווג הקוונטי, כלומר `QuantumEnhancedEnsembleClassifier`. | לא | `QuantumEnhancedEnsembleClassifier` |
| `num_learners` | `integer` | מספר הלומדים באנסמבל. | לא | `10` |
| `learners_types` | `list` | סוגי הלומדים. בין הסוגים הנתמכים: `DecisionTreeClassifier`, `GaussianNB`, `KNeighborsClassifier`, `MLPClassifier` ו-`LogisticRegression`. פרטים נוספים על כל אחד ניתן למצוא ב[תיעוד scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | לא | `[DecisionTreeClassifier]` |
| `learners_proportions` | `list` | יחסי כל סוג לומד באנסמבל. | לא | `[1.0]` |
| `learners_options` | `list` | אפשרויות לכל סוג לומד באנסמבל. לרשימה מלאה של האפשרויות המתאימות לסוג הלומד שנבחר, ראה ב[תיעוד scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | לא | `[{"max_depth": 3, "splitter": "random", "class_weight": None}]` |
| `regularization_type` | `str` או `list` | סוג/ים של רגולריזציה לשימוש: `onsite` או `alpha`. `onsite` שולט בגורם ה-onsite כאשר ערכים גבוהים יותר מובילים לאנסמבלים דלילים יותר. `alpha` שולט בפשרה בין גורמי האינטראקציה וה-onsite כאשר ערכים נמוכים יותר מובילים לאנסמבלים דלילים יותר. אם מסופקת רשימה, מודלים יאומנו לכל סוג והמוצלח ביותר ייבחר. | לא | `onsite` |
| `regularization` | `str` או `float` או `list` | ערך הרגולריזציה. מוגבל בין `0` ל-`+inf` אם regularization_type הוא `onsite`. מוגבל בין `0` ל-`1` אם regularization_type הוא `alpha`. אם מוגדר ל-`auto`, נעשה שימוש ברגולריזציה אוטומטית — פרמטר הרגולריזציה האופטימלי נמצא בחיפוש בינארי עם יחס הסיווגים הנבחרים לסיווגים הכולל הרצוי (`regularization_desired_ratio`) והגבול העליון לפרמטר הרגולריזציה (`regularization_upper_bound`). אם מסופקת רשימה, מודלים יאומנו לכל ערך והמוצלח ביותר ייבחר. | לא | `0.01` |
| `regularization_desired_ratio` | `float` או `list` | יחס/ים רצויים של סיווגים נבחרים לסיווגים כולל לרגולריזציה אוטומטית. אם מסופקת רשימה, מודלים יאומנו לכל יחס והמוצלח ביותר ייבחר. | לא | `0.75` |
| `regularization_upper_bound` | `float` או `list` | גבול/ות עליון/ים לפרמטר הרגולריזציה בעת שימוש ברגולריזציה אוטומטית. אם מסופקת רשימה, מודלים יאומנו לכל גבול עליון והמוצלח ביותר ייבחר. | לא | `200` |
| `weight_update_method` | `str` | שיטה לעדכון משקלי דגימה מבין `logarithmic` ו-`quadratic`. | לא | `logarithmic` |
| `sample_scaling` | `boolean` | האם להחיל קנה מידה על הדגימות. | לא | `False` |
| `prediction_scaling` | `float` | גורם קנה מידה לתחזיות. | לא | `None` |
| `optimizer_options` | `dictionary` | אפשרויות האופטימייזר של QAOA. רשימת האפשרויות הזמינות מוצגת בהמשך התיעוד. | לא | ... |
| `voting` | `str` | השתמש בהצבעת רוב (`hard`) או ממוצע הסתברויות (`soft`) לאגרגציה של תחזיות/הסתברויות הלומדים. | לא | `hard` |
| `prob_threshold` | `float` | סף הסתברות אופטימלי. | לא | `0.5` |
| `random_state` | `integer` | שליטה באקראיות לצורך ניתן לחזרה. | לא | `None` |


- בנוסף, `optimizer_options` מפורטות כדלקמן:

|     שם    |    סוג     | תיאור | חובה  | ברירת מחדל |
|-------------|-------------|-------------|-----------|---------|
| `num_solutions` | `integer` | מספר הפתרונות | לא | `1024` |
| `reps` | `integer` | מספר החזרות | לא | `4` |
| `sparsify` | `float` | סף הדילול | לא | `0.001` |
| `theta` | `float` | הערך ההתחלתי של theta, פרמטר וריאציוני של QAOA | לא | `None` |
| `simulator` | `boolean` | האם להשתמש בסימולטור או ב-QPU | לא | `False` |
| `classical_optimizer` | `str` | שם האופטימייזר הקלאסי ל-QAOA. כל ה-solvers המוצעים על ידי SciPy, כמפורט [כאן](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html#scipy.optimize.minimize), ניתנים לשימוש. יש להגדיר את `classical_optimizer_options` בהתאם | לא | `COBYLA` |
| `classical_optimizer_options` | `dictionary` | אפשרויות האופטימייזר הקלאסי. לרשימה מלאה של האפשרויות הזמינות, ראה ב[תיעוד SciPy](https://docs.scipy.org/doc/scipy/reference/) | לא | `{"maxiter": 60}` |
| `optimization_level` | `integer` | עומק ה-Circuit של QAOA | לא | `3` |
| `num_transpiler_runs` | `integer` | מספר ריצות ה-Transpiler | לא | `30` |
| `pass_manager_options` | `dictionary` | אפשרויות ליצירת pass manager מוגדר מראש | לא | `{"approximation_degree": 1.0}` |
| `estimator_options` | `dictionary` | אפשרויות Estimator. לרשימה מלאה של האפשרויות הזמינות, ראה ב[תיעוד Qiskit Runtime Client](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) | לא | `None` |
| `sampler_options` | `dictionary` | אפשרויות Sampler. לרשימה מלאה של האפשרויות הזמינות, ראה ב[תיעוד Qiskit Runtime Client](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options) | לא | `None` |

- ברירות המחדל של `estimator_options` הן:

|     שם    |    סוג     | ערך  |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `2` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |
| `resilience_options` | `dictionary` | `{"zne_mitigation": False, "zne": {"amplifier": "pea", "noise_factors": [1.0, 1.3, 1.6], "extrapolator": ["linear", "polynomial_degree_2", "exponential"],}}` |

- ברירות המחדל של `sampler_options` הן:

|     שם    |    סוג     | ערך |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `1` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |

#### שימוש

In [4]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [5]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}


['X_test.npy.tar', 'X_train.npy.tar', 'my_classifier.pkl.tar', 'y_test.npy.tar', 'y_train.npy.tar']


#### אימותים
- `name`:
    - השם חייב להיות ייחודי, מחרוזת באורך עד 64 תווים.
    - יכול לכלול רק תווים אלפאנומריים וקווים תחתונים.
    - חייב להתחיל באות ולא יכול להסתיים בקו תחתון.
    - מסווג עם אותו שם חייב כבר לקיים בספריית הנתונים המשותפת.
### 4. Fit

פעולת `fit` מאמנת מסווג באמצעות נתוני האימון שסופקו.

#### קלט
|     שם    |    סוג     | תיאור |   חובה  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | שם הפעולה. חייב להיות `fit`. | כן |
| `name`      | `str`    | שם המסווג לאימון. | כן |
| `X`         | `array` או `list` או `str` | נתוני האימון. יכול להיות מערך NumPy, רשימה, או מחרוזת המפנה לשם קובץ בספריית הנתונים המשותפת. | כן |
| `y`         | `array` או `list` או `str` | ערכי היעד לאימון. יכול להיות מערך NumPy, רשימה, או מחרוזת המפנה לשם קובץ בספריית הנתונים המשותפת. | כן |
| `fit_params`| `dictionary`| פרמטרים נוספים להעברה לשיטת `fit` של המסווג. | לא |

##### fit_params
|     שם    |    סוג     | תיאור |   חובה  |   ברירת מחדל   |
|-------------|-------------|-------------|-------------|-------------|
| `validation_data` | `tuple` | נתוני האימות והתוויות. | לא | `None` |
| `pos_label` | `integer` או `str` | תווית המחלקה שתמופה ל-1. | לא | `None` |
| `optimization_data` | `str` | סט נתונים לאופטימיזציה של האנסמבל עליו. יכול להיות אחד מ: `train`, `validation`, `both`. | לא | `train` |

#### שימוש

In [6]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 13.655871629714966}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 54.688621282577515}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 56.92286920547485}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 57.92738223075867}}}}


#### אימותים
- `name`:
    - השם חייב להיות ייחודי, מחרוזת באורך עד 64 תווים.
    - יכול לכלול רק תווים אלפאנומריים וקווים תחתונים.
    - חייב להתחיל באות ולא יכול להסתיים בקו תחתון.
    - מסווג עם אותו שם חייב כבר לקיים בספריית הנתונים המשותפת.
### 5. Predict

פעולת `predict` משמשת לקבלת תחזיות קשות ורכות (הסתברויות).

#### קלט
|     שם    |    סוג     | תיאור |   חובה  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | שם הפעולה. חייב להיות `predict`. | כן |
| `name`      | `str`    | שם המסווג לשימוש. | כן |
| `X`         | `array` או `list` או `str` | נתוני הבדיקה. יכול להיות מערך NumPy, רשימה, או מחרוזת המפנה לשם קובץ בספריית הנתונים המשותפת. | כן |
| `options["out"]` | `str` | שם קובץ JSON הפלט לשמירת התחזיות בספריית הנתונים המשותפת. אם לא סופק, התחזיות מוחזרות בתוצאת ה-job. | לא |

#### שימוש

In [7]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 0, 1, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]]


#### אימותים
- `name`:
    - השם חייב להיות ייחודי, מחרוזת באורך עד 64 תווים.
    - יכול לכלול רק תווים אלפאנומריים וקווים תחתונים.
    - חייב להתחיל באות ולא יכול להסתיים בקו תחתון.
    - מסווג עם אותו שם חייב כבר לקיים בספריית הנתונים המשותפת.
- `options["out"]`:
    - שם הקובץ חייב להיות ייחודי, מחרוזת באורך עד 64 תווים.
    - יכול לכלול רק תווים אלפאנומריים וקווים תחתונים.
    - חייב להתחיל באות ולא יכול להסתיים בקו תחתון.
    - חייב לכלול את הסיומת `.json`.
### 6. Fit-predict

פעולת `fit_predict` מאמנת מסווג באמצעות נתוני האימון ולאחר מכן משתמשת בו לקבלת תחזיות קשות ורכות (הסתברויות).

#### קלט
|     שם    |    סוג     | תיאור |   חובה  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | שם הפעולה. חייב להיות `fit_predict`. | כן |
| `name`      | `str`    | שם המסווג לשימוש. | כן |
| `X_train`   | `array` או `list` או `str` | נתוני האימון. יכול להיות מערך NumPy, רשימה, או מחרוזת המפנה לשם קובץ בספריית הנתונים המשותפת. | כן |
| `y_train`   | `array` או `list` או `str` | ערכי היעד לאימון. יכול להיות מערך NumPy, רשימה, או מחרוזת המפנה לשם קובץ בספריית הנתונים המשותפת. | כן |
| `X_test`    | `array` או `list` או `str` | נתוני הבדיקה. יכול להיות מערך NumPy, רשימה, או מחרוזת המפנה לשם קובץ בספריית הנתונים המשותפת. | כן |
| `fit_params`| `dictionary`| פרמטרים נוספים להעברה לשיטת `fit` של המסווג. | לא |
| `options["out"]` | `str` | שם קובץ JSON הפלט לשמירת התחזיות בספריית הנתונים המשותפת. אם לא סופק, התחזיות מוחזרות בתוצאת ה-job. | לא |

#### שימוש

In [8]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


#### אימותים
- `name`:
    - השם חייב להיות ייחודי, מחרוזת באורך עד 64 תווים.
    - יכול לכלול רק תווים אלפאנומריים וקווים תחתונים.
    - חייב להתחיל באות ולא יכול להסתיים בקו תחתון.
    - מסווג עם אותו שם חייב כבר לקיים בספריית הנתונים המשותפת.

- `options["out"]`:
    - שם הקובץ חייב להיות ייחודי, מחרוזת באורך עד 64 תווים.
    - יכול לכלול רק תווים אלפאנומריים וקווים תחתונים.
    - חייב להתחיל באות ולא יכול להסתיים בקו תחתון.
    - חייב לכלול את הסיומת `.json`.
### 7. Create-fit-predict

פעולת `create_fit_predict` יוצרת מסווג, מאמנת אותו באמצעות נתוני האימון שסופקו, ולאחר מכן משתמשת בו לקבלת תחזיות קשות ורכות (הסתברויות).

#### קלט
| שם | סוג | תיאור | חובה |
|------|------|-------------|-----------|
| `action` | `str` | שם הפעולה מבין `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` ו-`delete`. | כן |
| `name` | `str` | שם המסווג לשימוש. | כן |
| `quantum_classifier` | `str` | סוג המסווג, כלומר `QuantumEnhancedEnsembleClassifier`. ברירת המחדל היא `QuantumEnhancedEnsembleClassifier`. | לא |
| `X_train` | `array` או `list` או `str` | נתוני האימון. יכול להיות מערך NumPy, רשימה, או מחרוזת המפנה לשם קובץ בספריית הנתונים המשותפת. | כן |
| `y_train` | `array` או `list` או `str` | ערכי היעד לאימון. יכול להיות מערך NumPy, רשימה, או מחרוזת המפנה לשם קובץ בספריית הנתונים המשותפת. | כן |
| `X_test` | `array` או `list` או `str` | נתוני הבדיקה. יכול להיות מערך NumPy, רשימה, או מחרוזת המפנה לשם קובץ בספריית הנתונים המשותפת. | כן |
| `fit_params` | `dictionary` | פרמטרים נוספים להעברה לשיטת `fit` של המסווג. | לא |
| `options["save"]` | `boolean` | האם לשמור את המסווג המאומן בספריית הנתונים המשותפת. ברירת המחדל היא `True`. | לא |
| `options["out"]` | `str` | שם קובץ JSON הפלט לשמירת התחזיות בספריית הנתונים המשותפת. אם לא סופק, התחזיות מוחזרות בתוצאת ה-job. | לא |

#### שימוש

In [9]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

'Requested file was deleted.'

#### אימותים
- `name`:
    - אם `options["save"]` מוגדר ל-`True`:
        - השם חייב להיות ייחודי, מחרוזת באורך עד 64 תווים.
        - יכול לכלול רק תווים אלפאנומריים וקווים תחתונים.
        - חייב להתחיל באות ולא יכול להסתיים בקו תחתון.
        - אסור שמסווג עם אותו שם כבר יקיים בספריית הנתונים המשותפת.

- `options["out"]`:
    - שם הקובץ חייב להיות ייחודי, מחרוזת באורך עד 64 תווים.
    - יכול לכלול רק תווים אלפאנומריים וקווים תחתונים.
    - חייב להתחיל באות ולא יכול להסתיים בקו תחתון.
    - חייב לכלול את הסיומת `.json`.
---
## התחלה
אמת את זהותך באמצעות [מפתח ה-API של IBM Quantum Platform](http://quantum.cloud.ibm.com/), ובחר את פונקציית Qiskit כך:

In [10]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [0, 0, 1, 0, 0]
Probabilities (first five results):  [[0.87119766531518, 0.1288023346848197], [0.87119766531518, 0.1288023346848197], [0.24470328446479797, 0.7552967155352032], [0.820524432250189, 0.17947556774981072], [0.6847610293419495, 0.31523897065805173]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 10.967791318893433}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 59.91712307929993}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 59.097386837005615}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 56.93338203430176}}


## דוגמה
בדוגמה זו, תשתמש בפונקציית "Singularity Machine Learning - Classification" כדי לסווג סט נתונים המורכב משני חצאי עיגולים בצורת ירח המשתלבים. סט הנתונים הוא סינתטי, דו-ממדי, ומתויג בתוויות בינאריות. הוא נוצר כך שיהיה מאתגר לאלגוריתמים כמו קיבוץ מבוסס-צנטרואיד וסיווג לינארי.
![סט נתוני ירחים](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
במהלך תהליך זה, תלמד כיצד ליצור את המסווג, לאמן אותו על נתוני האימון, להשתמש בו לביצוע תחזיות על נתוני הבדיקה, ולמחוק את המסווג כשתסיים.
לפני שתתחיל, עליך להתקין את [scikit-learn](https://scikit-learn.org/). התקן אותו באמצעות הפקודה הבאה:
2) Upload the generated synthetic dataset to the [shared data directory](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).